# Notebook: Gibson cloning

## What this notebook does
- Demonstrates how to simulate PCR reactions (with and without overhangs) based on user-defined primers.  
- Generates PCR fragments from sequence templates and annotates them with useful metadata (primers, anneal length, circularity, etc.).  
- Uses these PCR products as input for a Gibson Assembly step to merge fragments into a final construct.  
- Exports the resulting assembled construct as a GenBank file, with standardized annotations and safe filenames.  

---

## Inputs & expected layout
- **Excel sheet(s)** defining:
  - `ConstructID`: unique identifier for each construct.  
  - `RolPrimer`, `FwdPrimer`, `Reve`: primer definitions.  
  - `Order`, `FragmentRole`: assembly order of fragments.  
  - Optional: `Circularize`, `MinOverlap`, `IncludeOverhangs`, `MinAnneal`.  
- **Sequence records** (`role_records`) mapping roles (e.g. BACKBONE, INSERT_A) to DNA templates (`SeqRecord` objects).  
- **Output folder** (`out_dir`) where reports and GenBank files are written.  

---

## Workflow outline (cells in this notebook)

### Imports & setup
- Load required packages (Biopython, pandas, helper functions).
- Define paths to Excel file, report folder, and template sequences.

### PCR simulation
- Read PCR definitions from Excel.  
- Run either `simulate_pcr` or `simulate_pcr_overhangs` depending on primer design.  
- Annotate each product with primer metadata and collect them for downstream use.  

### Gibson assembly
- For each construct, order the PCR fragments as defined in Excel.  
- Merge them using `merge_with_gibson_features`, applying overlap and circularization rules.  
- Build a descriptive label for the construct and annotate the final `SeqRecord`.  

### Export & reporting
- Write the assembled construct(s) as GenBank file(s) to `out_dir`.  
- Print a summary of construct IDs, lengths, and output file names.  





In [ ]:
# ------------------------------ Imports & setup

from __future__ import annotations

from pathlib import Path
import os  # optional; keep if you use e.g. os.environ / os.listdir
import re
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import pandas as pd

from Bio import SeqIO, pairwise2
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.SeqFeature import SeqFeature, FeatureLocation, CompoundLocation

# Project API (top-level re-exports)
from assembly_designer import (
    # PCR & Gibson
    PCRResult,
    simulate_pcr,
    simulate_pcr_overhangs,
    merge_with_gibson_features,
    # Naming
    build_construct_label,
    #history
)

# Project API (submodule-only: not re-exported at top level)
from assembly_designer.plasmidio import (
    load_dna_file,
    remove_near_duplicate_features,
    _safe_filename,
)

# Optional dependency sanity check:
# - dnacauldron: required to simulate assemblies & write *_report.zip
# - snapgene_reader: enables reading .dna (SnapGene) files
try:
    import dnacauldron  # noqa: F401
    from snapgene_reader import snapgene_file_to_dict  # noqa: F401
    print("Optional deps OK: dnacauldron, snapgene-reader")
except Exception as err:
    print("⚠️ Optional deps check:", err)



In [ ]:
# ------------------------------ small helpers 
def _as_bool(x) -> bool:
    if isinstance(x, bool):
        return x
    if x is None:
        return False
    return str(x).strip().lower() in {"true", "1", "yes", "y", "ja"}

def _get_int(x, default: int) -> int:
    try:
        v = int(x)
        return v if v > 0 else default
    except Exception:
        return default
    
def _prefer(paths: List[Path], token: str) -> Path:
    for p in paths:
        if token.lower() in p.stem.lower():
            return p
    return paths[0]


## 1. PCR 

In [ ]:
# ------------------------------ Paths & loading 

BASE_DIR = Path.cwd()
ASSEMBLY_DIR = BASE_DIR / "reports" / "Assembly"
assert ASSEMBLY_DIR.is_dir(), f"Assembly folder not found: {ASSEMBLY_DIR}"

# Load TU constructs (deterministic pick by UNS pair if available)
tu_paths: List[Path] = sorted(ASSEMBLY_DIR.glob("*.gb*"))
assert len(tu_paths) >= 2, f"Expected at least 2 TU .gb/.gbk files in {ASSEMBLY_DIR}"

In [ ]:
# ------------------------------ Load all .gb/.gbk in the assembly folder (there should be exactly two final TUs)
tu_records: Dict[str, SeqRecord] = {}
for p in sorted(ASSEMBLY_DIR.glob("*.gb*")):
    rec = SeqIO.read(str(p), "genbank")
    stem = p.stem
    rec.id = stem
    rec.name = stem
    rec.annotations.setdefault("molecule_type", "DNA")
    tu_records[stem] = rec

print("Found TU constructs:", list(tu_records.keys()))
assert len(tu_records) >= 2, "Expected 2 TU constructs in the Assembly folder."

In [ ]:
# ------------------------------ Load data (no biology-specific logic)
BASE_DIR = Path.cwd()
ASSEMBLY_DIR = BASE_DIR / "reports" / "Assembly"
if not ASSEMBLY_DIR.is_dir():
    raise FileNotFoundError(f"Assembly folder not found: {ASSEMBLY_DIR}")

tu_paths: List[Path] = sorted(ASSEMBLY_DIR.glob("*.gb*"))
if len(tu_paths) < 2:
    raise FileNotFoundError(f"Expected at least 2 TU .gb/.gbk files in {ASSEMBLY_DIR}")

tu1_path = _prefer(tu_paths, "UNS1A_UNS3_E")
tu2_path = _prefer(tu_paths, "UNS3A_UNS10E")

tu1_rec: SeqRecord = SeqIO.read(str(tu1_path), "genbank")
tu2_rec: SeqRecord = SeqIO.read(str(tu2_path), "genbank")
for rec, p in [(tu1_rec, tu1_path), (tu2_rec, tu2_path)]:
    rec.id = rec.name = p.stem
    rec.annotations.setdefault("molecule_type", "DNA")

print("TUs:", tu1_rec.id, "|", tu2_rec.id)


In [ ]:
# ------------------------------ Search for backboene vector file (prefer .dna if available)
CANDIDATES = [BASE_DIR / "Gibson_Backbone_parts",
              BASE_DIR / "Backbone_Parts",
              BASE_DIR / "Backbone_parts"]
vec_dir = next((d for d in CANDIDATES if d.is_dir()), None)

vec_files = [p for p in vec_dir.iterdir() if p.suffix.lower() in {".dna", ".gb", ".gbk", ".genbank"}]
if not vec_files:
    raise FileNotFoundError(f"No vector files in {vec_dir}")

vector_path = next((p for p in vec_files if p.suffix.lower() == ".dna"), vec_files[0])
vector_rec = load_dna_file(vector_path)

vector_rec.id = vector_rec.name = vector_path.stem
vector_rec.annotations.setdefault("molecule_type", "DNA")
print("Vector:", vector_rec.id)

In [ ]:
# ------------------------------ Load Excel "gibson_designs_template.xlsx" and extract relevant sheets
xlsx_path = "gibson_designs_template.xlsx" 
xl = pd.read_excel(xlsx_path, sheet_name=None)

constructs_df = xl.get("Constructs", pd.DataFrame())
pcr_df        = xl["PCR"].copy()
assembly_df   = xl["Assembly"].copy()

# vector_rec, tu1_rec, tu2_rec
role_records: Dict[str, SeqRecord] = {
    "Backbone": vector_rec,
    "TU1": tu1_rec,
    "TU2": tu2_rec,
}

constructs_df


In [ ]:
pcr_df.head()

In [ ]:
# ------------------------------ Generate all PCR products (Backbone supports overhangs) ---
products: Dict[Tuple[str, str], SeqRecord] = {}
pcr_results: Dict[Tuple[str, str], PCRResult] = {}  # keep full results if you want

# Context:
# - pcr_df: DataFrame with PCR setup (columns like ConstructID, RolPrimer, Reve, FwdPrimer, Circular, IncludeOverhangs, MinAnneal)
# - role_records: mapping from role name -> SeqRecord (template sequence for that role)
# - simulate_pcr / simulate_pcr_overhangs: PCR simulators returning a PCRResult with .product (SeqRecord) and metadata
# - _as_bool / _get_int: helpers to safely parse booleans/ints from mixed inputs
# - products / pcr_results: dicts collecting outputs, keyed by (ConstructID, Role)

for _, row in pcr_df.iterrows():
    # Read per-row configuration and normalize fields
    cid  = str(row["ConstructID"])                                   # project/construct identifier
    role = str(row["Role"]).replace(" ", "")                    # role/category (e.g., Backbone, Insert); strip spaces
    rev  = str(row["RevPrimer"])                                          # reverse primer (5'→3'; will be RC-searched)
    fwd  = str(row["FwdPrimer"]).replace(" ", "")                    # forward primer (5'→3'); strip spaces
    circ = _as_bool(row.get("Circular", True))                        # treat template as circular? default True
    ovh  = _as_bool(row.get("IncludeOverhangs", role.upper() == "BACKBONE"))
                                                                      # include 5' overhangs? default True for BACKBONE
    min_anneal = _get_int(row.get("MinAnneal", 18), 18)               # minimal annealing length (for overhang mode)

    # Fetch the template SeqRecord for this role; try exact, then UPPER
    template = role_records.get(role) or role_records.get(role.upper())
    if template is None:
        raise KeyError(f"No sequence record mapped for role '{role}'.")

    # Construct a readable product ID for downstream files/plots
    pid = f"{cid}_{role}_PCR"

    # Choose simulation mode:
    # - Overhang mode: finds longest unique anneal (>= min_anneal), preserves/annotates 5' overhangs
    # - Exact mode: primers must match exactly (no overhang logic)
    if ovh:
        res = simulate_pcr_overhangs(
            template,
            fwd_primer=fwd,
            rev_primer=rev,
            circular=circ,
            min_anneal=min_anneal,
            include_overhangs=True,
            product_id=pid,     # set explicit product ID
        )
    else:
        res = simulate_pcr(
            template,
            fwd_primer=fwd,
            rev_primer=rev,
            circular=circ,
        )
        # Ensure a consistent product ID for exact mode as well
        res.product.id = res.product.name = pid

    product = res.product

    # Attach PCR meta so provenance/history/plots can read parameters directly from the product
    product.annotations["pcr_meta"] = {
        "construct_id": cid,
        "role": role,
        "fwd_primer": fwd,
        "rev_primer": rev,
        "circular": bool(circ),
        "include_overhangs": bool(ovh),
        "min_anneal": int(min_anneal),
        "fwd_anneal_len": getattr(res, "fwd_anneal_len", None),  # only present in overhang mode
        "rev_anneal_len": getattr(res, "rev_anneal_len", None),  # only present in overhang mode
    }

    # Collect outputs for later export/plotting
    products[(cid, role)] = product       # final SeqRecord (PCR product)
    pcr_results[(cid, role)] = res        # full PCRResult (optional but useful for indices/lengths)

print(f"PCR done for {len(products)} (ConstructID, Role) pairs.")


In [ ]:
# output example:
products[('C1','TU2')]

## 2. Gibson

In [ ]:
# ------------------------------ output dir
out_dir = Path("reports") / "3G"
out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# ------------------------------ Optional: load TUs sheet
tus_df = xl.get("TUs", pd.DataFrame())

finals: Dict[str, SeqRecord] = {}

In [ ]:
assembly_df

In [ ]:
# === Goal =========================================================
# Assemble a single construct with Gibson assembly,
# using the PCR products already present in `products`.
# Produces one final SeqRecord, writes a GenBank file, and stores it in `finals`.
# ==================================================================

# ------------------------------------------------------------------
# Optional: set the construct explicitly (recommended)
# If you omit this, the code will infer the only ConstructID in assembly_df.
# ------------------------------------------------------------------
TARGET_CID = None  # e.g., "C1"

# ------------------------------------------------------------------
# 1) Pick the single ConstructID (either explicit or inferred)
# ------------------------------------------------------------------
if TARGET_CID is None:
    # Infer the only ConstructID present; raises if there are 0 or >1
    uniq = assembly_df["ConstructID"].dropna().unique().tolist()
    if len(uniq) != 1:
        raise ValueError(f"Expected exactly one ConstructID in assembly_df, found: {uniq}")
    cid = uniq[0]
else:
    cid = TARGET_CID

# ------------------------------------------------------------------
# 2) Extract the rows for this construct and derive fragment order/roles
# ------------------------------------------------------------------
grp = assembly_df.loc[assembly_df["ConstructID"] == cid]
if grp.empty:
    raise KeyError(f"No rows in assembly_df for ConstructID='{cid}'.")

# Sort by the 'Order' column and pull the roles in assembly order
order: List[str] = grp.sort_values("Order")["FragmentRole"].tolist()

# Map roles to their PCR products (SeqRecord) — no loop, just a list comprehension
# (Comprehension is allowed here as an expression; no explicit for-loop block.)
frags = [products[(cid, role)] for role in order]

# ------------------------------------------------------------------
# 3) Read construct-level defaults (min overlap & circularization)
#    If missing, fall back to sensible defaults.
# ------------------------------------------------------------------
rowc = constructs_df.loc[constructs_df["ConstructID"] == cid]
min_ov = (
    _get_int(rowc["MinOverlap"].iloc[0], 40)
    if (not rowc.empty and "MinOverlap" in rowc)
    else 40
)
circ = (
    _as_bool(rowc["Circularize"].iloc[0])
    if (not rowc.empty and "Circularize" in rowc)
    else True
)

# ------------------------------------------------------------------
# 4) Perform Gibson merge (features preserved by your helper)
# ------------------------------------------------------------------
final = merge_with_gibson_features(
    frags,
    min_overlap=min_ov,
    circularize=circ,
)

# ------------------------------------------------------------------
# 5) Build a human-readable label for the construct
# ------------------------------------------------------------------
label, part_labels = build_construct_label(
    cid=cid,
    order=order,
    constructs_df=constructs_df,
    tus_df=tus_df,
    prefix_with_cid=True,    # keep "C1__" prefix
    include_uns_in_tu=True,  # append UNS context if available
)

# ------------------------------------------------------------------
# 6) Annotate the final record (IDs kept short for GenBank), then write file
# ------------------------------------------------------------------
final.id = label[:20]                            # short GenBank ID
final.name = final.id
final.description = f"{label} (Gibson assembly)"
final.annotations.setdefault("molecule_type", "DNA")
final.annotations.setdefault("accessions", [final.id])
final.annotations.setdefault("sequence_version", 1)

# Safe filename and write to disk
out_path = out_dir / f"{_safe_filename(label)}.gb"
SeqIO.write(final, out_path, "genbank")

# ------------------------------------------------------------------
# 7) Store in `finals` dict and report
# ------------------------------------------------------------------
finals[cid] = final
print(f"✅ {cid}: {len(final)} bp → {out_path.name}")


In [ ]:
# if you have more than one construct in that excel sheet, repeat the above block for each ConstructID

# ------------------------------ Gibson assembly of all constructs

for cid, grp in assembly_df.groupby("ConstructID"):
    # Fragment order and PCR products for this construct
    order = grp.sort_values("Order")["FragmentRole"].tolist()
    frags = [products[(cid, role)] for role in order]

    # Defaults from Constructs (if present)
    rowc = constructs_df[constructs_df["ConstructID"] == cid]
    min_ov = _get_int(rowc["MinOverlap"].iloc[0], 40) if (not rowc.empty and "MinOverlap" in rowc) else 40
    circ   = _as_bool(rowc["Circularize"].iloc[0])     if (not rowc.empty and "Circularize" in rowc) else True

    # Assemble
    final = merge_with_gibson_features(frags, min_overlap=min_ov, circularize=circ)

    # Build descriptive label using the public helper
    label, part_labels = build_construct_label(
        cid=cid,
        order=order,
        constructs_df=constructs_df,
        tus_df=tus_df,
        prefix_with_cid=True,   # keep "C1__" prefix
        include_uns_in_tu=True, # append UNS context if available
    )

    # Set record identifiers (keep GenBank id short)
    final.id = label[:20]
    final.name = final.id
    final.description = f"{label} (Gibson assembly)"
    # Ensure minimal annotations for GenBank writers/viewers
    final.annotations.setdefault("molecule_type", "DNA")
    final.annotations.setdefault("accessions", [final.id])
    final.annotations.setdefault("sequence_version", 1)

    # Save with a safe, descriptive filename
    out_path = out_dir / f"{_safe_filename(label)}.gb"
    SeqIO.write(final, out_path, "genbank")

    finals[cid] = final
    print(f"✅ {cid}: {len(final)} bp → {out_path.name}")



In [ ]:
# ------------------------------ final outputas dictionary 
finals